# 4.0 — Baselines clasificación multiclase (nonMalignant vs tipos de cáncer)

Este notebook replica el flujo de la iteración binaria, pero con etiquetas multiclase:
- `nonMalignant` (todas las patologías no-cáncer agrupadas)
- `Patient_group` para muestras `Malignant` (tipo de cáncer)

Además, se usa un **umbral sobre p(cáncer)** (`1 - p(nonMalignant)`) para controlar falsos negativos.

In [2]:
from __future__ import annotations
from pathlib import Path
from dataclasses import replace
import logging
import warnings
from sklearn.exceptions import ConvergenceWarning

import pandas as pd
import numpy as np

from time import perf_counter
from tqdm.auto import tqdm

from genomics_dl.models.train_multiclass import MulticlassTrainConfig, run_training

warnings.filterwarnings("ignore", category=ConvergenceWarning)

logging.getLogger("alembic").setLevel(logging.ERROR)
logging.getLogger("alembic.runtime.migration").setLevel(logging.ERROR)

logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("sqlalchemy").setLevel(logging.ERROR)
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    message=".*invalid value encountered in divide.*",
)

### Rutas y carga de datos

In [3]:
# Paths
DATA_PROCESSED = Path("../data/processed")
TRAIN_PATH = DATA_PROCESSED / "gse183635_tep_tpm_train.parquet"
TEST_PATH  = DATA_PROCESSED / "gse183635_tep_tpm_test.parquet"

df_train = pd.read_parquet(TRAIN_PATH)
df_test  = pd.read_parquet(TEST_PATH)

df_train.shape, df_test.shape

((1880, 5452), (471, 5452))

### Separación genes vs metadatos

In [4]:
metadata_cols = [
    "Sample ID",
    "Patient_group",
    "Stage",
    "Sex",
    "Age",
    "Sample-supplying institution",
    "Training series",
    "Evaluation series",
    "Validation series",
    "lib.size",
    "classificationScoreCancer",
    "Class_group",
]

# Genes: columnas ENSG...
gene_cols = [c for c in df_train.columns if str(c).startswith("ENSG")]

assert "Class_group" in df_train.columns
assert "Patient_group" in df_train.columns
assert len(gene_cols) > 0
assert set(gene_cols).isdisjoint(set(metadata_cols))

len(gene_cols), gene_cols[:5]

(5440,
 ['ENSG00000000419',
  'ENSG00000000460',
  'ENSG00000000938',
  'ENSG00000001036',
  'ENSG00000001461'])

### Sanity check de etiquetas (train)

In [5]:
df_train["Class_group"].value_counts(dropna=False)

Class_group
Malignant       1302
nonMalignant     578
Name: count, dtype: int64

In [6]:
df_train.loc[df_train["Class_group"].astype(str) == "Malignant", "Patient_group"].value_counts().head(20)

Patient_group
Non-small-cell lung cancer    417
Ovarian cancer                114
Glioma                        113
Pancreatic cancer              93
Breast cancer                  80
Head and neck cancer           79
Cholangiocarcinoma             71
Colorectal cancer              69
Melanoma                       54
Sarcoma                        44
Endometrial cancer             34
Prostate cancer                23
Multiple Myeloma               22
Urothelial cancer              22
Renal cell cancer              20
Hepatocellular carcinoma       19
Lymphoma                       16
Esophageal carcinoma           12
Name: count, dtype: int64

## Experimentos baseline (sweep)

Ranking:
1) Minimizar `test_cancer_fn` (cáncer predicho como nonMalignant)
2) Maximizar `test_cancer_recall_sensitivity`
3) Maximizar `test_f1_macro`

In [7]:
def slugify_token(value):
    return str(value).replace(".", "p").replace("-", "m")

def build_model_name(clf_name, feat_cfg, malignant_weight, variant_tag):
    return "_".join([
        clf_name,
        f"pca{int(feat_cfg['use_pca'])}",
        f"log{int(feat_cfg['selector_on_log'])}",
        f"vq{int(feat_cfg['var_quantile']*100)}",
        f"mw{slugify_token(malignant_weight)}",
        variant_tag,
    ])

def fmt_secs(s: float) -> str:
    s = int(max(0, s))
    h = s // 3600
    m = (s % 3600) // 60
    ss = s % 60
    if h > 0:
        return f"{h:d}h {m:02d}m {ss:02d}s"
    if m > 0:
        return f"{m:d}m {ss:02d}s"
    return f"{ss:d}s"

In [8]:
# Base config
base_cfg = MulticlassTrainConfig(
    train_path=str(TRAIN_PATH),
    test_path=str(TEST_PATH),
    model_name="multiclass_smoke",
    model_version="v0.1.0",
    use_pca=False,
    var_quantile=0.2,
    selector_on_log=False,
    pca_var_threshold=0.9,
    cv_splits=8,
    min_cancer_recall_for_threshold=0.9,
    threshold_objective="specificity",
    experiment_name="gse183635_multiclass_smoke",
    save_local_bundle=False,
    save_plots=False,
)

# Sweep
feat_grid = [
    dict(use_pca=False, selector_on_log=False, var_quantile=0.2, pca_var_threshold=0.9),
    dict(use_pca=False, selector_on_log=True, var_quantile=0.2, pca_var_threshold=0.9),
    dict(use_pca=True, selector_on_log=False, var_quantile=0.2, pca_var_threshold=0.9),
]

clf_grid = [
    ("logreg", dict(solver="lbfgs", max_iter=6000)),
    ("sgd_logloss", dict(alpha=1e-3, max_iter=6000, tol=1e-3, early_stopping=True, validation_fraction=0.1, n_iter_no_change=5, average=True,)),
    ("linear_svc_calibrated", dict(C=1.0)),
    ("extratrees", dict(n_estimators=800)),
]

malignant_weights = [1.0, 2.0, 4.0]

sweep = []
for feat_cfg in feat_grid:
    for clf_name, clf_params in clf_grid:
        for mw in malignant_weights:
            sweep.append(dict(feat_cfg=feat_cfg, clf_name=clf_name, clf_params=clf_params, mw=mw))

len(sweep)

36

In [9]:
results = []
errors = []

total = len(sweep)
start_all = perf_counter()

# Calculadora de tiempo por iteración y estimación con media móvil
ema = None
alpha = 0.25
done = 0

pbar = tqdm(sweep, total=total, desc="Sweep", unit="run")

for combo in pbar:
    t0 = perf_counter()

    feat_cfg = combo["feat_cfg"]
    clf_name = combo["clf_name"]
    clf_params = combo["clf_params"]
    mw = combo["mw"]

    model_name = build_model_name(clf_name, feat_cfg, mw, variant_tag="sweep")
    cfg = replace(
        base_cfg,
        model_name=model_name,
        clf_name=clf_name,
        clf_params=clf_params,
        malignant_weight=mw,
        use_pca=feat_cfg["use_pca"],
        selector_on_log=feat_cfg["selector_on_log"],
        var_quantile=feat_cfg["var_quantile"],
        pca_var_threshold=feat_cfg["pca_var_threshold"],
    )

    try:
        out = run_training(cfg, feature_cols=gene_cols)
        tm = out["test_metrics"]
        results.append({
            "model_name": model_name,
            "clf_name": clf_name,
            "mw": mw,
            "use_pca": feat_cfg["use_pca"],
            "selector_on_log": feat_cfg["selector_on_log"],
            "var_quantile": feat_cfg["var_quantile"],
            "test_cancer_fn": tm["cancer_fn"],
            "test_cancer_fnr": tm["cancer_fnr"],
            "test_cancer_recall": tm["cancer_recall_sensitivity"],
            "test_f1_macro": tm["f1_macro"],
            "test_accuracy": tm["accuracy"],
            "mlflow_run_id": out["mlflow_run_id"],
        })
    except Exception as e:
        errors.append({
            "model_name": model_name,
            "clf_name": clf_name,
            "mw": mw,
            "use_pca": feat_cfg["use_pca"],
            "selector_on_log": feat_cfg["selector_on_log"],
            "var_quantile": feat_cfg["var_quantile"],
            "error": repr(e),
        })

    dt = perf_counter() - t0
    ema = dt if ema is None else (alpha * dt + (1 - alpha) * ema)

    done += 1
    elapsed = perf_counter() - start_all
    remaining = (total - done) * (ema if ema is not None else 0.0)

    pbar.set_postfix({
        "last": fmt_secs(dt),
        "avg": fmt_secs(ema),
        "elapsed": fmt_secs(elapsed),
        "eta": fmt_secs(remaining),
        "ok": len(results),
        "err": len(errors),
    })

# DataFrames finales
res_df = (
    pd.DataFrame(results)
      .sort_values(["test_cancer_fn", "test_cancer_fnr", "test_cancer_recall", "test_f1_macro"],
                   ascending=[True, True, False, False])
      .reset_index(drop=True)
)

err_df = pd.DataFrame(errors).reset_index(drop=True)

print("OK:", len(res_df), "Errores:", len(err_df))

Sweep: 100%|██████████| 36/36 [1:23:04<00:00, 138.45s/run, last=38s, avg=1m 38s, elapsed=1h 23m 04s, eta=0s, ok=27, err=9]        

OK: 27 Errores: 9


In [10]:
display(res_df)

,model_name,clf_name,mw,use_pca,selector_on_log,var_quantile,test_cancer_fn,test_cancer_fnr,test_cancer_recall,test_f1_macro,test_accuracy,mlflow_run_id
0,extratrees_pca1_log0_vq20_mw4p0_sweep,extratrees,4.0,True,False,0.2,31,0.095092,0.904908,0.121683,0.405520,a040c6513e0e43c9a025e396a95dbe11
1,extratrees_pca1_log0_vq20_mw1p0_sweep,extratrees,1.0,True,False,0.2,37,0.113497,0.886503,0.116023,0.424628,9ffd1051fd174a0ab21cdbe84e59a88e
2,extratrees_pca0_log0_vq20_mw2p0_sweep,extratrees,2.0,False,False,0.2,40,0.122699,0.877301,0.159432,0.443737,9ce05522deef468f8badb86949a5d71d
3,extratrees_pca0_log0_vq20_mw1p0_sweep,extratrees,1.0,False,False,0.2,40,0.122699,0.877301,0.155139,0.437367,855b2816e4654d64a083a2c3f808b382
4,logreg_pca0_log0_vq20_mw1p0_sweep,logreg,1.0,False,False,0.2,41,0.125767,0.874233,0.350555,0.560510,c60eb539fc524ff7ae7bee61e8bfb82b
5,extratrees_pca0_log0_vq20_mw4p0_sweep,extratrees,4.0,False,False,0.2,41,0.125767,0.874233,0.167136,0.441614,b7b0d9f526b04e98a661a0660d046c8e
6,extratrees_pca0_log1_vq20_mw1p0_sweep,extratrees,1.0,False,True,0.2,41,0.125767,0.874233,0.133156,0.428875,d692f22e161640bdb704336cf8eea68e
7,extratrees_pca1_log0_vq20_mw2p0_sweep,extratrees,2.0,True,False,0.2,41,0.125767,0.874233,0.125584,0.435244,d315275b012f4b1682112e1170e2572a
8,linear_svc_calibrated_pca1_log0_vq20_mw1p0_sweep,linear_svc_calibrated,1.0,True,False,0.2,41,0.125767,0.874233,0.056725,0.392781,e931bec62d564671b5124acbc49b31ac
9,logreg_pca1_log0_vq20_mw2p0_sweep,logreg,2.0,True,False,0.2,42,0.128834,0.871166,0.356090,0.520170,11df4448242c4872a824f99367243b43


In [11]:
if len(err_df) > 0:
    display(err_df)

,model_name,clf_name,mw,use_pca,selector_on_log,var_quantile,error
0,sgd_logloss_pca0_log0_vq20_mw1p0_sweep,sgd_logloss,1.0,False,False,0.2,ValueError('Input contains NaN.')
1,sgd_logloss_pca0_log0_vq20_mw2p0_sweep,sgd_logloss,2.0,False,False,0.2,ValueError('Input contains NaN.')
2,sgd_logloss_pca0_log0_vq20_mw4p0_sweep,sgd_logloss,4.0,False,False,0.2,ValueError('Input contains NaN.')
3,sgd_logloss_pca0_log1_vq20_mw1p0_sweep,sgd_logloss,1.0,False,True,0.2,ValueError('Input contains NaN.')
4,sgd_logloss_pca0_log1_vq20_mw2p0_sweep,sgd_logloss,2.0,False,True,0.2,ValueError('Input contains NaN.')
5,sgd_logloss_pca0_log1_vq20_mw4p0_sweep,sgd_logloss,4.0,False,True,0.2,ValueError('Input contains NaN.')
6,sgd_logloss_pca1_log0_vq20_mw1p0_sweep,sgd_logloss,1.0,True,False,0.2,ValueError('Input contains NaN.')
7,sgd_logloss_pca1_log0_vq20_mw2p0_sweep,sgd_logloss,2.0,True,False,0.2,ValueError('Input contains NaN.')
8,sgd_logloss_pca1_log0_vq20_mw4p0_sweep,sgd_logloss,4.0,True,False,0.2,ValueError('Input contains NaN.')


## Entrenamiento final (guardar bundle en `models/`)

In [12]:
# Elegimos el mejor del sweep
best = res_df.iloc[0].to_dict()
best

{'model_name': 'extratrees_pca1_log0_vq20_mw4p0_sweep',
 'clf_name': 'extratrees',
 'mw': 4.0,
 'use_pca': True,
 'selector_on_log': False,
 'var_quantile': 0.2,
 'test_cancer_fn': 31,
 'test_cancer_fnr': 0.0950920245398773,
 'test_cancer_recall': 0.9049079754601227,
 'test_f1_macro': 0.12168254974146642,
 'test_accuracy': 0.40552016985138006,
 'mlflow_run_id': 'a040c6513e0e43c9a025e396a95dbe11'}

In [13]:
best_cfg = replace(
    base_cfg,
    model_name="multiclass_final",
    model_version="v0.2.0",
    clf_name=best["clf_name"],
    malignant_weight=float(best["mw"]),
    use_pca=bool(best["use_pca"]),
    selector_on_log=bool(best["selector_on_log"]),
    var_quantile=float(best["var_quantile"]),
    save_local_bundle=True,
    save_plots=True,
)

final_out = run_training(best_cfg, feature_cols=gene_cols)
final_out

{'mlflow_run_id': '794cefb66e2c463eb9ace332a977f084',
 'cv_metrics': {'accuracy': 0.4175531914893617,
  'balanced_accuracy': 0.12644781975274558,
  'f1_macro': 0.13320114221073429,
  'f1_weighted': 0.355471939991926,
  'log_loss': 2.0144846350909766,
  'cancer_threshold': 0.608,
  'cancer_tn': 306,
  'cancer_fp': 272,
  'cancer_fn': 126,
  'cancer_tp': 1176,
  'cancer_fnr': 0.0967741935483871,
  'cancer_recall_sensitivity': 0.9032258064516129,
  'cancer_specificity': 0.5294117647058824,
  'cancer_precision': 0.8121546961325967,
  'cancer_roc_auc': 0.8309866641153616,
  'cancer_pr_auc': 0.901997428492447,
  'per_class_report': {'Breast cancer': {'precision': 0.7058823529411765,
    'recall': 0.15,
    'f1-score': 0.24742268041237114,
    'support': 80.0},
   'Cholangiocarcinoma': {'precision': 0.0,
    'recall': 0.0,
    'f1-score': 0.0,
    'support': 71.0},
   'Colorectal cancer': {'precision': 1.0,
    'recall': 0.028985507246376812,
    'f1-score': 0.056338028169014086,
    'support